<a href="https://colab.research.google.com/github/oliver-morris/Dictionary-Language-Model/blob/main/NeuralNetZero.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dictionary-Based LLM

Attempt at making a more computationally efficient LLM by leveraging dictionary-basedf learning.

# Imports

In [1]:
!pip install -q transformers datasets sentencepiece nltk

In [2]:
import random
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import nltk
from nltk.corpus import wordnet as wn
from transformers import AutoTokenizer

nltk.download("wordnet")
nltk.download("omw-1.4")

device = "cuda" if torch.cuda.is_available() else "cpu"

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


In [30]:
print(torch.cuda.memory_allocated()/1024**3, "GB allocated")
print(torch.cuda.memory_reserved()/1024**3, "GB reserved")

torch.cuda.empty_cache()

print(torch.cuda.memory_allocated()/1024**3, "GB allocated")
print(torch.cuda.memory_reserved()/1024**3, "GB reserved")

10.71357774734497 GB allocated
11.24609375 GB reserved
10.71357774734497 GB allocated
11.24609375 GB reserved


In [3]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
MAX_LEN = 64

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

# Dataset

In [4]:
samples = []

for syn in wn.all_synsets():
    word = syn.lemmas()[0].name().replace("_", " ")
    definition = syn.definition()
    examples = syn.examples()
    # Task 1
    samples.append(
        (
            f"define: {word}",
            definition
        )
    )

    # Task 2
    samples.append(
        (
            f"word: {definition}",
            word
        )
    )

    # Task 3
    if len(examples) > 0:
        samples.append(
            (
                f"example: {word}",
                examples[0]
            )
        )

random.shuffle(samples)
print("Training samples:", len(samples))

Training samples: 268241


In [5]:
encoded = []

for inp, out in samples:

    x = tokenizer(
        inp,
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN
    )["input_ids"]

    y = tokenizer(
        out,
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN
    )["input_ids"]

    encoded.append((torch.tensor(x), torch.tensor(y)))

In [36]:
class DictDataset(Dataset):
    def __init__(self, pairs, encoded):
        self.encoded = encoded
        self.data = pairs

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.encoded[idx]

dataset = DictDataset(samples, encoded)

loader = DataLoader(
    dataset,
    batch_size=64,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

In [37]:
x, y = next(iter(loader))

print(tokenizer.decode(x[0]))
print(tokenizer.decode(y[0]))

[CLS] define : botswana monetary unit [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]
[CLS] monetary unit in botswana [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]


# Transformer

In [38]:
VOCAB = tokenizer.vocab_size

class TinyDictionaryTransformer(nn.Module):

    def __init__(self):

        super().__init__()

        self.embedding = nn.Embedding(VOCAB,256)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=256,
            nhead=4,
            dim_feedforward=512,
            dropout=0.1,
            batch_first=True
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=6
        )

        self.fc = nn.Linear(256,VOCAB)

    def forward(self,x):

        x = self.embedding(x)

        x = self.encoder(x)

        return self.fc(x)

model = TinyDictionaryTransformer().to(device)

# Training

In [39]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4
)

criterion = nn.CrossEntropyLoss(
    ignore_index=tokenizer.pad_token_id
)

In [40]:
EPOCHS = 5

scaler = torch.amp.GradScaler("cuda")

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    for x, y in loader:

        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            output = model(x)

            loss = criterion(
                output.view(-1, VOCAB),
                y.view(-1)
            )

        scaler.scale(loss).backward()

        scaler.step(optimizer)

        scaler.update()

        total_loss += loss.item()


    avg_loss = total_loss / len(loader)

    print(
        f"Epoch {epoch+1}/{EPOCHS} Loss: {avg_loss:.4f}"
    )

Epoch 1/5 Loss: 6.0011
Epoch 2/5 Loss: 5.7596
Epoch 3/5 Loss: 5.6009
Epoch 4/5 Loss: 5.4488
Epoch 5/5 Loss: 5.3011


In [41]:
model = torch.compile(model)

# Evaluating

In [44]:
model.eval()

while True:

    text = input("\nEnter prompt: ")

    if text=="quit":
        break

    tokens = tokenizer(
        text,
        return_tensors="pt",
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN
    ).input_ids.to(device)

    with torch.no_grad():

        out = model(tokens)

        ids = out.argmax(-1)

    print(
        tokenizer.decode(
            ids[0],
            skip_special_tokens=True
        )
    )


Enter prompt: quit
